# NFL Prospect Bin Scoring — v1 (Proportion Scoring)

**Goal:** For each player, measure what proportion of their scouting language falls into three bins:
1. **Physical** — size, speed, athleticism, body traits
2. **Technique/Skills** — football-specific craft, stamina, energy
3. **Character/IQ** — effort, intelligence, decision-making, intangibles

**Method:** Keyword proportion scoring. After preprocessing, count how many tokens match each bin's keyword set, then divide by total matched tokens → three proportions that sum to 100%.

**Pipeline:**
1. Phrase stitching (curated, NFL-aware)
2. Stopword removal (custom, NFL-aware)
3. Lemmatization
4. Dictionary seeding (manual anchor keywords per bin)
5. Word2Vec expansion (grow keyword lists using corpus semantics)
6. Proportion scoring (keyword token counting)
7. Output: proportions + top matched keywords per player

In [ ]:
import pandas as pd
import numpy as np
import re
import json
from collections import Counter
from itertools import combinations

from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

# ── Config ─────────────────────────────────────────────────────────────────────
W2V_DIM       = 100   # embedding dimensions
W2V_WINDOW    = 6     # context window
W2V_EPOCHS    = 30    # training passes
W2V_SG        = 1     # 1 = skip-gram
W2V_MIN_COUNT = 3     # ignore tokens appearing fewer than N times

SEED_TOPN      = 20   # W2V neighbors to retrieve per seed
SIM_THRESHOLD  = 0.35 # min cosine similarity for expansion candidates
MIN_SEED_COUNT = 2    # candidate must appear near >= N seeds to qualify
MAX_W2V_TERMS  = 25   # max W2V-expanded terms added per bin

TOP_TERMS_N = 5       # top matched keywords to surface per player-bin pair

BIN_NAMES = ['physical', 'technique', 'character']

FILTER_SPECIALISTS = True   # set False to include K, P, LS in all analysis

print('Imports OK')

## Section 1 — Load Data

In [ ]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')

keep_cols = ['player_name', 'Pos_Group', 'position', 'grade', 'year',
             'made_it_contract', 'overview', 'strengths', 'weaknesses']
df = df[keep_cols].copy()

# Fill missing text — keep all rows, never drop
for col in ['overview', 'strengths', 'weaknesses']:
    df[col] = df[col].fillna('')

print(f'Players: {len(df)}')
print(f'Positions: {df["position"].nunique()} unique')
print(f'\nText coverage:')
for col in ['overview', 'strengths', 'weaknesses']:
    pct = (df[col].str.strip() != '').mean() * 100
    print(f'  {col:12s}: {pct:.1f}%')
# ── Specialist filter ───────────────────────────────────────────────────────────
if FILTER_SPECIALISTS:
    n_before = len(df)
    df = df[df['Pos_Group'] != 'SPECIAL'].reset_index(drop=True)
    print(f'Specialists removed: {n_before - len(df)}  (K, P, LS — Pos_Group=SPECIAL)')
    print(f'Players remaining  : {len(df)}')
else:
    print('Specialists included.')

## Section 2 — Preprocessing

Three steps applied in order:
1. **Phrase stitching** — replace multi-word NFL terms with underscore tokens *before* any tokenization, so `route running` → `route_running` survives as a single unit
2. **Stopword removal** — custom NFL-aware list; preserves football-relevant words like `high`, `low`, `hard`, `soft` that standard NLTK stops would remove
3. **Lemmatization** — reduces inflected forms to base (`rushing` → `rush`, `blocks` → `block`)

*PMI bigram discovery removed from v3 — curated phrases only, for simplicity and reproducibility.*

In [ ]:
# ── Curated compound NFL terms ─────────────────────────────────────────────────
# Sorted longest-first to prevent partial matches (trigrams before bigrams)
_CURATED_RAW = {
    # Trigrams & longer
    'change of direction':    'change_of_direction',
    'low pad level':          'low_pad_level',
    'run after catch':        'run_after_catch',
    'yards after contact':    'yards_after_contact',
    'yards after catch':      'yards_after_catch',
    'off the line':           'off_the_line',
    'off the ball':           'off_the_ball',
    'point of attack':        'point_of_attack',
    'get off the line':       'get_off_the_line',
    'read and react':         'read_and_react',
    'yards after catch':      'yards_after_catch',
    
    # Bigrams
    'pass rush':              'pass_rush',
    'pass rusher':            'pass_rusher',
    'pass protection':        'pass_protection',
    'pass coverage':          'pass_coverage',
    'pad level':              'pad_level',
    'press coverage':         'press_coverage',
    'man coverage':           'man_coverage',
    'zone coverage':          'zone_coverage',
    'zone coverages':         'zone_coverages',
    'ball skills':            'ball_skills',
    'ball hawk':              'ball_hawk',
    'ball hawking':           'ball_hawking',
    'ball carrier':           'ball_carrier',
    'ball carriers':          'ball_carriers',
    'body control':           'body_control',
    'contact balance':        'contact_balance',
    'closing speed':          'closing_speed',
    'lateral quickness':      'lateral_quickness',
    'quick twitch':           'quick_twitch',
    'high motor':             'high_motor',
    'first step':             'first_step',
    'get off':                'get_off',
    'hand fighting':          'hand_fighting',
    'hand strength':          'hand_strength',
    'hand placement':         'hand_placement',
    'strong hands':           'strong_hands',
    'soft hands':             'soft_hands',
    'heavy hands':            'heavy_hands',
    'block shedding':         'block_shedding',
    'anchor strength':        'anchor_strength',
    'route running':          'route_running',
    'run blocking':           'run_blocking',
    'open field':             'open_field',
    'red zone':               'red_zone',
    'second level':           'second_level',
    'hip flexibility':        'hip_flexibility',
    'short area':             'short_area',
    'three down':             'three_down',
    'top end':                'top_end',
    'two gap':                'two_gap',
    'one gap':                'one_gap',
    'two gapping':            'two_gapping',
    'one gapping':            'one_gapping',
    'snap count':             'snap_count',
    'football iq':            'football_iq',
    'film study':             'film_study',
    'work ethic':             'work_ethic',
    'locker room':            'locker_room',
    'decision making':        'decision_making',
    'play making':            'play_making',
    'field vision':           'field_vision',
    'play recognition':       'play_recognition',
    'pre snap':               'pre_snap',
    'post snap':              'post_snap',
    'game ready':             'game_ready',
    'hard worker':            'hard_worker',
    'arm_length':             'arm_length',
    'functional strength':    'functional_strength',
    'tackling form':          'tackling_form',
    'processing speed':       'processing_speed',
    'eye discipline':         'eye_discipline',
    'spatial awareness':      'spatial_awareness',
    'route progressions':     'route_progressions',
}

CURATED_PHRASE_MAP = dict(
    sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)
print(f'Curated phrases: {len(CURATED_PHRASE_MAP)}')

# ── NFL-aware stop words ────────────────────────────────────────────────────────
# Preserve football-relevant words that NLTK would remove
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great',
    'up', 'down', 'off', 'out', 'over', 'through', 'above', 'below',
    'hand', 'hands', 'back', 'field', 'ball' # Added these to prevent losing 'soft hands' or 'open field'
}

CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside',
    'starter', 'backup', 'senior', 'junior', 'cb', 'rb', 'wr', 'qb' # Added position noise filters
}

_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS

print(f'Base NLTK stops  : {len(_base)}')
print(f'Un-stopped        : {len(KEEP_WORDS & _base)}  (kept from NLTK list)')
print(f'Custom added      : {len(CUSTOM_STOPS)}')
print(f'Final stop list   : {len(NFL_STOPWORDS)}')

In [ ]:
lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str, phrase_map: dict = CURATED_PHRASE_MAP) -> str:
    """NFL-aware preprocessing: normalize → stitch → clean → filter → lemmatize."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)       # normalize all dash variants
    for phrase, token in phrase_map.items():              # curated phrase stitching
        text = text.replace(phrase, token)
    text = re.sub(r'[^a-z_\s]', ' ', text)              # keep letters + underscores only
    tokens = text.split()
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)

# Apply to each section
df['strengths_clean']  = df['strengths'].apply(nfl_preprocess)
df['overview_clean']   = df['overview'].apply(nfl_preprocess)
df['weaknesses_clean'] = df['weaknesses'].apply(nfl_preprocess)

# Combine all sections into one token list per player (Option A — section-agnostic)
df['all_text']   = (df['overview_clean'] + ' ' +
                    df['strengths_clean'] + ' ' +
                    df['weaknesses_clean']).str.strip()
df['all_tokens'] = df['all_text'].apply(str.split)

lengths = df['all_tokens'].apply(len)
print(f'Preprocessing complete.')
print(f'Tokens per player — median: {lengths.median():.0f},  min: {lengths.min()},  max: {lengths.max()}')

## Section 3 — Word2Vec Training

W2V is trained on all player text. Its **sole role** in this pipeline is **vocabulary expansion** in the next section — it identifies which corpus terms cluster near our manual seed words, extending keyword lists beyond what we could hand-curate.

W2V output is never used for scoring. Scoring is done by simple keyword token counting.

In [ ]:
sentences = df['all_tokens'].tolist()

w2v = Word2Vec(
    sentences   = sentences,
    vector_size = W2V_DIM,
    window      = W2V_WINDOW,
    epochs      = W2V_EPOCHS,
    sg          = W2V_SG,
    min_count   = W2V_MIN_COUNT,
    workers     = 4,
    seed        = 42,
)

print(f'W2V trained.  Vocab size: {len(w2v.wv)} tokens')

# Quick sanity — check a few seeds are in vocab
spot_check = ['explosive', 'footwork', 'football_iq', 'route_running', 'toughness']
for t in spot_check:
    status = '✓' if t in w2v.wv else '✗ MISSING'
    print(f'  {status}  {t}')

## Section 4 — Dictionary Seeding & W2V Expansion

**Step 1 — Anchor seeds:** small, unambiguous, high-frequency keyword lists per bin.
These are hand-curated to be clearly representative of each bin with minimal cross-bin overlap.

**Step 2 — W2V expansion:** for each seed, retrieve nearest neighbors from the trained corpus.
Keep candidates that appear near ≥ 2 seeds (multi-seed confirmation reduces noise).

**Step 3 — Manual additions:** stitched compound phrases (`route_running`, `football_iq`) and
domain-critical terms that W2V may miss due to low frequency or stitching.

**Step 4 — Overlap check:** flag any term appearing in multiple bins and resolve.

In [ ]:
# ── Anchor seeds — minimal, unambiguous, high-frequency ────────────────────────
# These seed W2V expansion. Keep lists tight and non-overlapping.
ANCHOR_SEEDS = {
    'physical': [
        'explosive', 'burst', 'speed', 'acceleration', 'first_step', 'get_off',
        'change_of_direction', 'agility', 'frame', 'size', 'strength',
        'power', 'athletic', 'physical', 'twitch',
    ],
    'technique': [
        'technique', 'footwork', 'leverage', 'pad_level', 'mechanics',
        'route_running', 'pass_protection', 'hand_fighting', 'block_shedding',
        'anchor_strength', 'pass_rush', 'blocking', 'tackling', 'coverage',
    ],
    'character': [
        'effort', 'motor', 'high_motor', 'relentless', 'competitive',
        'toughness', 'instinct', 'awareness', 'intelligence', 'football_iq',
        'coachable', 'discipline', 'leadership', 'work_ethic', 'recognition',
    ],
}

# Vocab check — flag missing seeds so we can adjust if needed
print('Anchor seed vocab coverage:')
for bin_name, seeds in ANCHOR_SEEDS.items():
    in_vocab = [s for s in seeds if s in w2v.wv]
    missing  = [s for s in seeds if s not in w2v.wv]
    print(f'  {bin_name:12s}: {len(in_vocab)}/{len(seeds)} in W2V vocab'
          + (f'  — MISSING: {missing}' if missing else ''))

In [ ]:
def expand_bin(seeds: list, model: Word2Vec,
               topn: int = SEED_TOPN,
               threshold: float = SIM_THRESHOLD) -> pd.DataFrame:
    """
    For each seed in W2V vocab, retrieve topN nearest neighbors.
    Aggregate by: how many seeds each candidate appears near (seed_count)
    and average similarity. Returns sorted DataFrame of candidates.
    """
    candidate_scores: dict = {}
    for seed in seeds:
        if seed not in model.wv:
            continue
        for word, sim in model.wv.most_similar(seed, topn=topn):
            if sim >= threshold and word not in seeds:
                candidate_scores.setdefault(word, []).append(sim)

    if not candidate_scores:
        return pd.DataFrame(columns=['term', 'seed_count', 'avg_sim', 'max_sim'])

    rows = [{'term': w,
             'seed_count': len(s),
             'avg_sim': round(float(np.mean(s)), 3),
             'max_sim': round(float(np.max(s)),  3)}
            for w, s in candidate_scores.items()]
    return (pd.DataFrame(rows)
              .sort_values(['seed_count', 'avg_sim'], ascending=False)
              .reset_index(drop=True))

# Run expansion for all three bins
BIN_LEXICONS = {}
for bin_name, seeds in ANCHOR_SEEDS.items():
    BIN_LEXICONS[bin_name] = expand_bin(seeds, w2v)

print('W2V expansion complete.')
print()
for bin_name, lex in BIN_LEXICONS.items():
    hc = lex[lex['seed_count'] >= MIN_SEED_COUNT]
    print(f'{bin_name} — {len(hc)} high-confidence expansions (seed_count >= {MIN_SEED_COUNT}):')
    print(hc.head(MAX_W2V_TERMS).to_string(index=False))
    print()

In [ ]:
# ── Manual additions ───────────────────────────────────────────────────────────
# Compound stitched phrases and domain-critical terms W2V may miss
# (low frequency, multi-word tokens, or terms below min_count threshold)
MANUAL_ADDITIONS = {
    'physical': [
        # Size & measurement language
        'height', 'length', 'wingspan', 'weight', 'build',
        'big', 'tall', 'wide', 'lean', 'heavy', 'mass', 'thick', 'long',
        'frame', 'arm_length', 'high_waisted', 'dense', 'lanky', 'stout',
        # Movement quality
        'body_control', 'fluidity', 'fluid', 'flexible', 'flexibility',
        'lateral_quickness', 'closing_speed', 'top_end', 'quick_twitch',
        # Athleticism markers
        'natural', 'raw', 'powerful', 'fast', 'athlete', 'dynamic',
        'shifty', 'explosion', 'vertical', 'leap', 'projectable',
        'explosiveness', 'twitchy', 'functional_strength', 'displacement',
        # Rarity language
        'rare', 'specimen', 'elite', 'scouts_dream'
    ],
    'technique': [
        # Compound craft terms (stitched — critical to preserve)
        'pass_rush', 'pass_rusher', 'pass_protection', 'route_running',
        'run_blocking', 'block_shedding', 'anchor_strength', 'hand_fighting',
        'press_coverage', 'zone_coverage', 'man_coverage', 'low_pad_level',
        'point_of_attack', 'ball_skills', 'soft_hands', 'heavy_hands',
        'hand_placement', 'punch', 'shed', 'leveraged', 'mirror',
        # Vision & Processing (Moved from Character to Technique/Skill)
        'vision', 'field_vision', 'spatial_awareness', 'anticipation', 
        'eye_discipline', 'read_and_react', 'processing_speed', 'route_progressions',
        # RB/WR/TE Skill Specifics
        'contact_balance', 'run_after_catch', 'yards_after_catch', 'yards_after_contact',
        'uncover', 'uncovering', 'salesmanship', 'get_off_the_line',
        # DB/Defensive Technique
        'pedal', 'backpedal', 'wrapping', 'tackling_form', 'zone_coverages',
        # Stamina / conditioning 
        'stamina', 'endurance', 'conditioning', 'energy', 'snap_count', 'three_down',
        # Craft descriptors
        'precise', 'sound', 'polished', 'refined', 'crisp', 'clean',
        'skilled', 'technical', 'fundamental', 'mechanics', 'craft',
        'skill', 'execution', 'assignment'
    ],
    'character': [
        # Effort & motor
        'high_motor', 'work_ethic', 'hard_worker', 'hustle', 'grit',
        'relentless', 'compete', 'pursuit', 'intensity', 'passion',
        'competitive_toughness', 'alpha',
        # Mental / IQ (Focusing on intangibles)
        'football_iq', 'decision_making', 'play_recognition', 'pre_snap',
        'post_snap', 'cerebral', 'smart', 'savvy', 'diagnose', 'read',
        'instinct', 'play_making',
        # Character markers
        'leadership', 'accountable', 'driven', 'focused', 'committed',
        'workhorse', 'coachable', 'film_study', 'locker_room', 'captain',
        'consistent', 'reliable', 'dependable', 'composure', 'mature',
        'dedicated', 'trustworthy', 'selfless', 'teammate', 'red_flag'
    ],
}

# ── Build final keyword sets ────────────────────────────────────────────────────
KEYWORD_SETS = {}
W2V_CONTRIBUTIONS = {}  # track what W2V added per bin (for transparency)

for bin_name in BIN_NAMES:
    seeds     = set(ANCHOR_SEEDS[bin_name])
    lex       = BIN_LEXICONS[bin_name]
    w2v_terms = set(lex[lex['seed_count'] >= MIN_SEED_COUNT]
                       .head(MAX_W2V_TERMS)['term'])
    manual    = set(MANUAL_ADDITIONS[bin_name])

    KEYWORD_SETS[bin_name]     = seeds | w2v_terms | manual
    W2V_CONTRIBUTIONS[bin_name] = w2v_terms - seeds - manual  # net-new from W2V only

print('Final keyword set sizes:')
for bin_name in BIN_NAMES:
    n_seed   = len(ANCHOR_SEEDS[bin_name])
    n_w2v    = len(W2V_CONTRIBUTIONS[bin_name])
    n_manual = len(MANUAL_ADDITIONS[bin_name])
    n_total  = len(KEYWORD_SETS[bin_name])
    print(f'  {bin_name:12s}: {n_total} terms  '
          f'(seeds: {n_seed}, W2V-only: {n_w2v}, manual: {n_manual})')

# ── Overlap check ───────────────────────────────────────────────────────────────
print()
found_overlap = False
for b1, b2 in combinations(BIN_NAMES, 2):
    overlap = KEYWORD_SETS[b1] & KEYWORD_SETS[b2]
    if overlap:
        print(f'⚠  Overlap {b1} ∩ {b2}: {sorted(overlap)}')
        found_overlap = True
if not found_overlap:
    print('✓  No keyword overlap between bins.')

## Section 5 — Proportion Scoring

For each player:
1. Iterate over their preprocessed token list
2. Count how many tokens match each bin's keyword set
3. Divide by total matched tokens → three proportions summing to 100%
4. Record which specific keywords drove each bin (top N by frequency)

Players with zero keyword matches (empty/very short reports) receive 0.0 for all bins.
Their `total_matched` column will be 0, making them easy to filter downstream.

In [ ]:
def score_player(tokens: list, keyword_sets: dict, top_n: int = TOP_TERMS_N) -> dict:
    """
    Count keyword matches per bin and return proportions + top matched terms.

    Parameters
    ----------
    tokens       : preprocessed token list for one player
    keyword_sets : {bin_name: set of keywords}
    top_n        : how many top matched keywords to return per bin

    Returns
    -------
    dict with _pct, _count, and _top_terms columns for each bin,
    plus total_matched
    """
    counts = {b: 0 for b in keyword_sets}
    hits   = {b: [] for b in keyword_sets}

    for token in tokens:
        for bin_name, keywords in keyword_sets.items():
            if token in keywords:
                counts[bin_name] += 1
                hits[bin_name].append(token)

    total = sum(counts.values())

    result = {'total_matched': total}

    for bin_name in keyword_sets:
        pct = counts[bin_name] / total if total > 0 else 0.0
        top = ', '.join(t for t, _ in Counter(hits[bin_name]).most_common(top_n))
        result[f'{bin_name}_pct']       = round(pct, 4)
        result[f'{bin_name}_count']     = counts[bin_name]
        result[f'{bin_name}_top_terms'] = top

    return result

# ── Apply to all players ────────────────────────────────────────────────────────
score_records = df['all_tokens'].apply(lambda toks: score_player(toks, KEYWORD_SETS))
scores_df     = pd.DataFrame(score_records.tolist())

ID_COLS = ['player_name', 'Pos_Group', 'position', 'grade', 'year', 'made_it_contract']
df_result = pd.concat([df[ID_COLS].reset_index(drop=True), scores_df], axis=1)

print(f'Scored {len(df_result)} players.')
print(f'\nColumn list: {list(df_result.columns)}')
print(f'\nSample (3 players):')
display_cols = ID_COLS + ['physical_pct', 'technique_pct', 'character_pct', 'total_matched']
print(df_result[display_cols].head(3).to_string(index=False))

## Section 6 — Validation & Spot Checks

In [ ]:
def player_report(name: str) -> None:
    """Print a detailed scoring breakdown for a named player."""
    matches = df_result[df_result['player_name'].str.lower() == name.lower()]
    if matches.empty:
        matches = df_result[df_result['player_name'].str.lower().str.contains(name.lower())]
    if matches.empty:
        print(f'Player "{name}" not found.')
        return

    row = matches.iloc[0]
    sep  = '=' * 55
    dash = '-' * 55
    print(sep)
    print(f'  {row["player_name"]}  |  {row["position"]}  |  Grade: {row["grade"]}')
    print(f'  Contract reached: {row["made_it_contract"]}')
    print(dash)
    for b in BIN_NAMES:
        pct   = row[f'{b}_pct'] * 100
        cnt   = row[f'{b}_count']
        terms = row[f'{b}_top_terms'] or '—'
        bar   = chr(9608) * int(pct / 5)
        print(f'  {b:12s} {pct:5.1f}%  {bar:<20s}  ({cnt} tokens)')
        print(f'               -> {terms}')
    print(dash)
    print(f'  total matched tokens: {row["total_matched"]}')
    print()

# Spot-check a handful of players
sample_names = df_result['player_name'].sample(5, random_state=42).tolist()
for name in sample_names:
    player_report(name)

In [ ]:
# ── Average bin proportions by position group ──────────────────────────────────
pos_summary = (
    df_result
    .groupby('Pos_Group')[['physical_pct', 'technique_pct', 'character_pct']]
    .mean()
    .multiply(100)
    .round(1)
    .rename(columns={
        'physical_pct':  'Physical %',
        'technique_pct': 'Technique %',
        'character_pct': 'Character %',
    })
    .sort_values('Physical %', ascending=False)
)

print('Average bin proportions by position group:')
print(pos_summary.to_string())

In [ ]:
# ── Coverage diagnostics ───────────────────────────────────────────────────────
zero_match = df_result['total_matched'] == 0
low_match  = df_result['total_matched'].between(1, 4)

print(f'Players with 0 keyword matches : {zero_match.sum()}')
print(f'Players with 1-4 matches       : {low_match.sum()}  (low-coverage, treat with caution)')
print(f'Players with ≥5 matches        : {(df_result["total_matched"] >= 5).sum()}')

if zero_match.sum() > 0:
    print('\nZero-match players (empty/very short reports):')
    print(df_result[zero_match][['player_name', 'position', 'grade']].to_string(index=False))

# Score distribution
print('\nBin score distribution (% of scouting language):')
print(df_result[['physical_pct', 'technique_pct', 'character_pct']]
      .multiply(100)
      .describe()
      .round(1)
      .to_string())

## Section 7 — Export

In [ ]:
# ── Main scores CSV ────────────────────────────────────────────────────────────
out_path = '../data/processed/bin_scores_v1.csv'
df_result.to_csv(out_path, index=False)
print(f'✓ Scores saved  →  {out_path}')
print(f'  Shape: {df_result.shape}')

# ── Keyword lexicon reference CSV ───────────────────────────────────────────────
# Records every term in each bin's final keyword set, tagged by source
lex_rows = []
for bin_name in BIN_NAMES:
    lex_df_idx = BIN_LEXICONS[bin_name].set_index('term') if len(BIN_LEXICONS[bin_name]) else pd.DataFrame()
    for term in sorted(KEYWORD_SETS[bin_name]):
        if term in ANCHOR_SEEDS[bin_name]:
            source = 'seed'
        elif term in MANUAL_ADDITIONS[bin_name]:
            source = 'manual'
        else:
            source = 'w2v'
        w2v_sim = (round(float(lex_df_idx.loc[term, 'avg_sim']), 3)
                   if (len(lex_df_idx) > 0 and term in lex_df_idx.index) else None)
        lex_rows.append({'bin': bin_name, 'term': term,
                         'source': source, 'w2v_avg_sim': w2v_sim})

lex_out = pd.DataFrame(lex_rows).sort_values(['bin', 'source', 'term'])
lex_path = '../data/processed/bin_lexicons_v1.csv'
lex_out.to_csv(lex_path, index=False)
print(f'✓ Lexicons saved →  {lex_path}')

print(f'\nLexicon summary:')
print(lex_out.groupby(['bin', 'source'])['term'].count().to_string())

## Section 8 — Temporal Trends by Position Group

Three line charts — one per bin — showing how the share of scouting language dedicated to each topic has shifted across eras.

- Years 2010 and 2011 are dropped (thin early sample)
- Remaining years split into 3 eras (configurable below)
- Each line = one position group; y-axis = mean % of scouting language
- Zero-match players excluded from aggregation

In [ ]:
# ── Year distribution — inspect before setting era cuts ───────────────────────
year_counts = df_result['year'].value_counts().sort_index()
print('Players per year (to inform era boundaries):')
print(year_counts.to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ── Era configuration — adjust cuts here if needed ─────────────────────────────
DROP_YEARS = [2010, 2011]
ERA_CUTS   = [2015, 2019]   # two cut points → three eras

ERA_LABELS = [
    f'Early (≤{ERA_CUTS[0]})',
    f'Mid ({ERA_CUTS[0]+1}–{ERA_CUTS[1]})',
    f'Recent ({ERA_CUTS[1]+1}+)',
]

def assign_era(year):
    if year <= ERA_CUTS[0]:
        return ERA_LABELS[0]
    elif year <= ERA_CUTS[1]:
        return ERA_LABELS[1]
    else:
        return ERA_LABELS[2]

# Filter dropped years, assign era
df_era = df_result[~df_result['year'].isin(DROP_YEARS)].copy()
df_era['era'] = df_era['year'].apply(assign_era)

print(f'Players after dropping {DROP_YEARS}: {len(df_era)}')
print()
print('Players per era:')
print(df_era.groupby('era')['player_name'].count().reindex(ERA_LABELS).to_string())

In [ ]:
# ── Aggregate: mean bin % per era × position group ─────────────────────────────
# Exclude zero-match players (no text → proportions are meaningless)
era_pos = (
    df_era[df_era['total_matched'] > 0]
    .groupby(['era', 'Pos_Group'])[['physical_pct', 'technique_pct', 'character_pct']]
    .mean()
    .multiply(100)
    .round(1)
    .reset_index()
)

# Preserve era order for plotting
era_order = {label: i for i, label in enumerate(ERA_LABELS)}
era_pos['era_order'] = era_pos['era'].map(era_order)
era_pos = era_pos.sort_values(['era_order', 'Pos_Group']).reset_index(drop=True)

print('Mean bin % by era and position group:')
print(era_pos.drop(columns='era_order').to_string(index=False))

In [ ]:
# ── 3-panel line chart — one panel per bin ──────────────────────────────────────
BIN_COLS   = ['physical_pct',  'technique_pct',    'character_pct']
BIN_TITLES = ['Physical',      'Technique / Skills', 'Character / IQ']
BIN_COLORS = ['#1565C0', '#E65100', '#2E7D32']   # deep blue, deep orange, deep green

pos_groups = sorted(era_pos['Pos_Group'].unique())

# Assign a consistent color per position group across all 3 panels
CMAP = plt.get_cmap('tab10')
pg_colors = {pg: CMAP(i) for i, pg in enumerate(pos_groups)}

# 1. CHANGED: set sharey=True for consistent scaling across panels
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# 2. OPTIONAL: Calculate a global max to ensure the labels have breathing room
global_max = era_pos[BIN_COLS].max().max()

for ax, col, title, accent in zip(axes, BIN_COLS, BIN_TITLES, BIN_COLORS):
    for pg in pos_groups:
        sub = (era_pos[era_pos['Pos_Group'] == pg]
               .sort_values('era_order'))
        if sub.empty:
            continue
        ax.plot(
            sub['era'], sub[col],
            marker='o', linewidth=2, markersize=7,
            color=pg_colors[pg], label=pg,
        )
        # Label the last point with pos group name
        last = sub.iloc[-1]
        ax.annotate(
            pg,
            xy=(last['era'], last[col]),
            xytext=(4, 0), textcoords='offset points',
            fontsize=7, va='center', color=pg_colors[pg],
        )

    # Styling
    ax.set_title(title, fontsize=14, fontweight='bold', color=accent, pad=10)
    ax.set_xlabel('Era', fontsize=10)
    
    # Only show Y label on the first plot to reduce clutter since they are shared
    if ax == axes[0]:
        ax.set_ylabel('Mean % of scouting language', fontsize=10)
    
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_xticks(range(len(ERA_LABELS)))
    ax.set_xticklabels(ERA_LABELS, rotation=10, ha='right', fontsize=9)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    
    # 3. CHANGED: Explicitly set y-limit starting at 0 and adding 10% headroom
    ax.set_ylim(bottom=0, top=global_max * 1.1)

# Shared legend below the figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    title='Position Group',
    loc='lower center',
    ncol=len(pos_groups),
    fontsize=9,
    title_fontsize=10,
    frameon=True,
    bbox_to_anchor=(0.5, -0.08),
)

plt.suptitle(
    'NFL Scouting Language Composition Over Time',
    fontsize=16, fontweight='bold', y=1.02,
)

# Use subplots_adjust to ensure title/legend/axes don't overlap with sharey
plt.tight_layout()

chart_path = '../data/processed/bin_trends_by_era.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved -> {chart_path}')

In [ ]:
# ── Filter and Aggregate: 2014-2026 & Grade >= 6.0 ────────────────────────────
# 1. Filter raw data
mask = (df_result['year'] >= 2014) & (df_result['year'] <= 2026) & (df_result['grade'] >= 6.0)
df_filtered = df_result[mask].copy()

# 2. Define Eras for the 2014-2026 window
ERA_CUTS   = [2017, 2021] 
ERA_LABELS = [
    f'2014–{ERA_CUTS[0]}',
    f'{ERA_CUTS[0]+1}–{ERA_CUTS[1]}',
    f'{ERA_CUTS[1]+1}+'
]

def assign_era(year):
    if year <= ERA_CUTS[0]: return ERA_LABELS[0]
    elif year <= ERA_CUTS[1]: return ERA_LABELS[1]
    else: return ERA_LABELS[2]

df_filtered['era'] = df_filtered['year'].apply(assign_era)

# 3. Create era_pos aggregation
era_pos = (
    df_filtered
    .groupby(['era', 'Pos_Group'])[['physical_pct', 'technique_pct', 'character_pct']]
    .mean()
    .multiply(100)
    .round(1)
    .reset_index()
)

# Assign order for plotting
era_order = {label: i for i, label in enumerate(ERA_LABELS)}
era_pos['era_order'] = era_pos['era'].map(era_order)
era_pos = era_pos.sort_values(['era_order', 'Pos_Group']).reset_index(drop=True)

# ── 3-panel line chart — one panel per bin ──────────────────────────────────────
BIN_COLS   = ['physical_pct',  'technique_pct',    'character_pct']
BIN_TITLES = ['Physical',      'Technique / Skills', 'Character / IQ']
BIN_COLORS = ['#1565C0', '#E65100', '#2E7D32']   # deep blue, deep orange, deep green

pos_groups = sorted(era_pos['Pos_Group'].unique())

# Assign a consistent color per position group across all 3 panels
CMAP = plt.get_cmap('tab10')
pg_colors = {pg: CMAP(i) for i, pg in enumerate(pos_groups)}

# 1. CHANGED: set sharey=True for consistent scaling across panels
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# 2. OPTIONAL: Calculate a global max to ensure the labels have breathing room
global_max = era_pos[BIN_COLS].max().max()

for ax, col, title, accent in zip(axes, BIN_COLS, BIN_TITLES, BIN_COLORS):
    for pg in pos_groups:
        sub = (era_pos[era_pos['Pos_Group'] == pg]
               .sort_values('era_order'))
        if sub.empty:
            continue
        ax.plot(
            sub['era'], sub[col],
            marker='o', linewidth=2, markersize=7,
            color=pg_colors[pg], label=pg,
        )
        # Label the last point with pos group name
        last = sub.iloc[-1]
        ax.annotate(
            pg,
            xy=(last['era'], last[col]),
            xytext=(4, 0), textcoords='offset points',
            fontsize=7, va='center', color=pg_colors[pg],
        )

    # Styling
    ax.set_title(title, fontsize=14, fontweight='bold', color=accent, pad=10)
    ax.set_xlabel('Era', fontsize=10)
    
    # Only show Y label on the first plot to reduce clutter since they are shared
    if ax == axes[0]:
        ax.set_ylabel('Mean % of scouting language', fontsize=10)
    
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_xticks(range(len(ERA_LABELS)))
    ax.set_xticklabels(ERA_LABELS, rotation=10, ha='right', fontsize=9)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    
    # 3. CHANGED: Explicitly set y-limit starting at 0 and adding 15% headroom for labels
    ax.set_ylim(bottom=0, top=global_max * 1.15)

# Shared legend below the figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    title='Position Group',
    loc='lower center',
    ncol=len(pos_groups),
    fontsize=9,
    title_fontsize=10,
    frameon=True,
    bbox_to_anchor=(0.5, -0.08),
)

plt.suptitle(
    'NFL Scouting Language Composition (2014-2026 | Grade >= 6.0)',
    fontsize=16, fontweight='bold', y=1.02,
)

plt.tight_layout()

chart_path = '../data/processed/bin_trends_modern_high_grade.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved -> {chart_path}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# 1. Filter: Modern Era (2014-2026) and High-Grade Prospects (>= 6.0)
mask = (df_result['year'] >= 2011) & (df_result['year'] <= 2025) & (df_result['grade'] >= 5.5)
df_yearly = df_result[mask].copy()

# 2. Aggregate: Mean per Year per Position
# We multiply by 100 here so the smoothing happens on the percentage scale
yearly_stats = (
    df_yearly
    .groupby(['year', 'Pos_Group'])[BIN_COLS]
    .mean()
    .multiply(100)
    .reset_index()
)

# 3. Apply Smoothing (3-year rolling average per Position Group)
# min_periods=1 ensures we don't lose the start/end years (2014 and 2026)
yearly_stats = yearly_stats.sort_values(['Pos_Group', 'year'])
for col in BIN_COLS:
    yearly_stats[f'{col}_smooth'] = (
        yearly_stats.groupby('Pos_Group')[col]
        .transform(lambda x: x.rolling(window=3, center=True, min_periods=1).mean())
    )

# ── 3-panel line chart — Yearly Smoothed ──────────────────────────────────────
BIN_TITLES = ['Physical', 'Technique / Skills', 'Character / IQ']
BIN_COLORS = ['#1565C0', '#E65100', '#2E7D32']
pos_groups = sorted(yearly_stats['Pos_Group'].unique())

# Consistent color mapping
CMAP = plt.get_cmap('tab10')
pg_colors = {pg: CMAP(i) for i, pg in enumerate(pos_groups)}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# Calculate global max for shared Y-axis headroom
global_max = yearly_stats[[f'{c}_smooth' for c in BIN_COLS]].max().max()

for ax, col, title, accent in zip(axes, BIN_COLS, BIN_TITLES, BIN_COLORS):
    smooth_col = f'{col}_smooth'
    
    for pg in pos_groups:
        sub = yearly_stats[yearly_stats['Pos_Group'] == pg]
        
        # Plot the smoothed line
        ax.plot(
            sub['year'], sub[smooth_col],
            marker=None, linewidth=3, alpha=0.9,
            color=pg_colors[pg], label=pg
        )
        
        # Optional: Plot the raw yearly dots at lower opacity to show the "truth"
        ax.scatter(
            sub['year'], sub[col],
            color=pg_colors[pg], s=15, alpha=0.2
        )
        
        # Annotate the final year
        last = sub.iloc[-1]
        ax.annotate(
            pg, xy=(last['year'], last[smooth_col]),
            xytext=(5, 0), textcoords='offset points',
            fontsize=7, va='center', color=pg_colors[pg], fontweight='bold'
        )

    # Styling
    ax.set_title(title, fontsize=14, fontweight='bold', color=accent, pad=15)
    ax.set_xlabel('Draft Year', fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel('Mean % of Scouting Language (3yr Smoothed)', fontsize=10)
    
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_xticks(range(2010, 2027, 2)) # Major ticks every 2 years
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    
    # Unified Y-limit with headroom
    ax.set_ylim(0, global_max * 1.15)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, title='Position Group',
    loc='lower center', ncol=len(pos_groups),
    bbox_to_anchor=(0.5, -0.1), frameon=True
)

plt.suptitle(
    'Temporal Evolution of Scouting Traits (2014-2026 | Ratings ≥ 6.0)',
    fontsize=16, fontweight='bold', y=1.05
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ── TOGGLE SETTINGS ──────────────────────────────────────────────────────────
# Change this to: 'Combined', 'Strengths', or 'Weaknesses'
TEXT_SOURCE = 'Strengths' 

# Global Filters
YEAR_RANGE = (2011, 2025)
MIN_GRADE  = 5.5
# ──────────────────────────────────────────────────────────────────────────────

# 1. Filter: Modern Era and High-Grade Prospects
# Note: We assume your dataframe has a 'text_type' or 'section' column 
# mapping to 'Strengths', 'Weaknesses', or 'Combined'
mask = (
    (df_result['year'] >= YEAR_RANGE[0]) & 
    (df_result['year'] <= YEAR_RANGE[1]) & 
    (df_result['grade'] >= MIN_GRADE)
)

# Apply the text source filter if applicable
if TEXT_SOURCE != 'Combined':
    # Adjust 'text_source_col' to your actual column name (e.g., 'section_name')
    mask = mask & (df_result['text_source'] == TEXT_SOURCE)

df_yearly = df_result[mask].copy()

# 2. Aggregate: Mean per Year per Position
yearly_stats = (
    df_yearly
    .groupby(['year', 'Pos_Group'])[BIN_COLS]
    .mean()
    .multiply(100)
    .reset_index()
)

# 3. Apply Smoothing (3-year rolling average per Position Group)
yearly_stats = yearly_stats.sort_values(['Pos_Group', 'year'])
for col in BIN_COLS:
    yearly_stats[f'{col}_smooth'] = (
        yearly_stats.groupby('Pos_Group')[col]
        .transform(lambda x: x.rolling(window=3, center=True, min_periods=1).mean())
    )

# ── 3-panel line chart ────────────────────────────────────────────────────────
BIN_TITLES = ['Physical', 'Technique / Skills', 'Character / IQ']
BIN_COLORS = ['#1565C0', '#E65100', '#2E7D32']
pos_groups = sorted(yearly_stats['Pos_Group'].unique())

CMAP = plt.get_cmap('tab10')
pg_colors = {pg: CMAP(i) for i, pg in enumerate(pos_groups)}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# Shared Y-axis headroom
smooth_cols = [f'{c}_smooth' for c in BIN_COLS]
global_max = yearly_stats[smooth_cols].max().max()

for ax, col, title, accent in zip(axes, BIN_COLS, BIN_TITLES, BIN_COLORS):
    smooth_col = f'{col}_smooth'
    
    for pg in pos_groups:
        sub = yearly_stats[yearly_stats['Pos_Group'] == pg]
        if sub.empty: continue
        
        ax.plot(sub['year'], sub[smooth_col], linewidth=3, alpha=0.9, color=pg_colors[pg], label=pg)
        ax.scatter(sub['year'], sub[col], color=pg_colors[pg], s=15, alpha=0.2)
        
        # Label the final point
        last = sub.iloc[-1]
        ax.annotate(pg, xy=(last['year'], last[smooth_col]), xytext=(5, 0), 
                    textcoords='offset points', fontsize=7, va='center', 
                    color=pg_colors[pg], fontweight='bold')

    # Styling
    ax.set_title(title, fontsize=14, fontweight='bold', color=accent, pad=15)
    ax.set_xlabel('Draft Year', fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel(f'Mean % of {TEXT_SOURCE} Language', fontsize=10)
    
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_xticks(range(YEAR_RANGE[0], YEAR_RANGE[1]+1, 2))
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, global_max * 1.15)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Position Group', loc='lower center', 
           ncol=len(pos_groups), bbox_to_anchor=(0.5, -0.1))

plt.suptitle(f'Temporal Evolution: {TEXT_SOURCE} Text ({YEAR_RANGE[0]}-{YEAR_RANGE[1]})', 
             fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()